# Support Vector Machines

An SVM finds the boundary that separates the classes with the **widest possible
margin** — the biggest gap to the nearest points of either class. Its superpower
is the **kernel trick**: swap the notion of "distance" and a linear boundary in a
transformed space becomes a curved boundary in the original one.

- a **linear** kernel gives a straight decision boundary;
- an **RBF** (radial basis function) kernel gives a flexible, curved one.

We visualise both on the 2-D breast-cancer slice (as in the
[KNN chapter](knn-classification.ipynb)), then run the full head-to-head of every
classifier in this part. `smartcore`'s `SVC` is a binary classifier — a good fit
for this two-class problem.

In [ ]:
:dep smartcore = { version = "0.3", features = ["datasets"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::dataset::breast_cancer;
use smartcore::model_selection::train_test_split;
use smartcore::metrics::{accuracy, f1};
use smartcore::svm::svc::{SVC, SVCParameters};
use smartcore::svm::Kernels;
use plotters::prelude::*;

// 2-D standardized view (features 0 & 1) for the boundary plots, plus the full
// 30-feature matrix for the accuracy comparison.
let (feat2, y, n): (Vec<f32>, Vec<i32>, usize) = {
    let ds = breast_cancer::load_dataset();
    let (n, p) = (ds.num_samples, ds.num_features);
    let yv: Vec<i32> = ds.target.iter().map(|&v| v as i32).collect();
    let mut mean = [0f64; 2];
    for i in 0..n { for j in 0..2 { mean[j] += ds.data[i * p + j] as f64; } }
    for j in 0..2 { mean[j] /= n as f64; }
    let mut sd = [0f64; 2];
    for i in 0..n { for j in 0..2 { let d = ds.data[i * p + j] as f64 - mean[j]; sd[j] += d * d; } }
    for j in 0..2 { sd[j] = (sd[j] / n as f64).sqrt(); }
    let mut feat2 = Vec::with_capacity(n * 2);
    for i in 0..n { for j in 0..2 { feat2.push(((ds.data[i * p + j] as f64 - mean[j]) / sd[j]) as f32); } }
    (feat2, yv, n)
};
let x2: DenseMatrix<f32> = DenseMatrix::new(n, 2, feat2.clone(), false);
// Standardize ALL 30 features for the comparison — distance/kernel methods
// (KNN, RBF-SVC) are scale-sensitive, so a fair fight needs comparable scales.
let x_full: DenseMatrix<f32> = {
    let ds = breast_cancer::load_dataset();
    let (nn, pp) = (ds.num_samples, ds.num_features);
    let mut mean = vec![0f64; pp];
    let mut sd = vec![0f64; pp];
    for i in 0..nn { for j in 0..pp { mean[j] += ds.data[i * pp + j] as f64; } }
    for j in 0..pp { mean[j] /= nn as f64; }
    for i in 0..nn { for j in 0..pp { let d = ds.data[i * pp + j] as f64 - mean[j]; sd[j] += d * d; } }
    for j in 0..pp { sd[j] = (sd[j] / nn as f64).sqrt(); if sd[j] == 0.0 { sd[j] = 1.0; } }
    let mut d = Vec::with_capacity(nn * pp);
    for i in 0..nn { for j in 0..pp { d.push(((ds.data[i * pp + j] as f64 - mean[j]) / sd[j]) as f32); } }
    DenseMatrix::new(nn, pp, d, false)
};
println!("{} samples ready (2-D view + full 30 features)", n);

## Linear vs RBF decision boundary

Same decision-region recipe as KNN: classify a grid of points and colour the
background. The **linear** kernel produces a straight frontier; the **RBF** kernel
bends around the data.

In [ ]:
let (mut x0, mut x1, mut y0, mut y1) = (f32::MAX, f32::MIN, f32::MAX, f32::MIN);
for i in 0..n {
    x0 = x0.min(feat2[i * 2]); x1 = x1.max(feat2[i * 2]);
    y0 = y0.min(feat2[i * 2 + 1]); y1 = y1.max(feat2[i * 2 + 1]);
}
let steps = 50usize;
let grid: DenseMatrix<f32> = {
    let mut g = Vec::with_capacity(steps * steps * 2);
    for gi in 0..steps { for gj in 0..steps {
        g.push(x0 + (x1 - x0) * gi as f32 / (steps - 1) as f32);
        g.push(y0 + (y1 - y0) * gj as f32 / (steps - 1) as f32);
    }}
    DenseMatrix::new(steps * steps, 2, g, false)
};
// smartcore's SVC needs labels in {-1, +1}, not {0, 1}.
let y_svc: Vec<i32> = y.iter().map(|&v| if v == 1 { 1 } else { -1 }).collect();

evcxr_figure((680, 340), |root| {
    root.fill(&WHITE)?;
    let panels = root.split_evenly((1, 2));
    let titles = ["linear kernel", "RBF kernel"];
    for (pi, panel) in panels.iter().enumerate() {
        // Each SVC borrows its params, so fit + predict inside this block; only the
        // owned Vec<i32> of grid predictions escapes.
        // smartcore's SVC::predict returns Vec<f32> (0.0/1.0), not the label type.
        let gpred: Vec<f32> = if pi == 0 {
            let params = SVCParameters::default().with_c(1.0f32).with_kernel(Kernels::linear());
            let model = SVC::fit(&x2, &y_svc, &params).unwrap();
            model.predict(&grid).unwrap()
        } else {
            let params = SVCParameters::default().with_c(1.0f32).with_kernel(Kernels::rbf().with_gamma(0.5));
            let model = SVC::fit(&x2, &y_svc, &params).unwrap();
            model.predict(&grid).unwrap()
        };
        let mut chart = ChartBuilder::on(panel)
            .caption(titles[pi], ("sans-serif", 15))
            .margin(5).x_label_area_size(24).y_label_area_size(28)
            .build_cartesian_2d(x0..x1, y0..y1)?;
        chart.configure_mesh().disable_mesh().draw()?;
        let cw = (x1 - x0) / (steps - 1) as f32;
        let ch = (y1 - y0) / (steps - 1) as f32;
        chart.draw_series((0..steps * steps).map(|idx| {
            let gi = (idx / steps) as f32;
            let gj = (idx % steps) as f32;
            let cx = x0 + (x1 - x0) * gi / (steps - 1) as f32;
            let cy = y0 + (y1 - y0) * gj / (steps - 1) as f32;
            let color = if gpred[idx] > 0.0 { RGBColor(255, 224, 189) } else { RGBColor(200, 220, 255) };
            Rectangle::new([(cx - cw / 2.0, cy - ch / 2.0), (cx + cw / 2.0, cy + ch / 2.0)], color.filled())
        }))?;
        chart.draw_series((0..n).step_by(2).map(|i| {
            let color = if y[i] == 1 { RGBColor(220, 120, 20) } else { RGBColor(30, 90, 200) };
            Circle::new((feat2[i * 2], feat2[i * 2 + 1]), 2, color.filled())
        }))?;
    }
    Ok(())
})

The linear frontier is a single straight cut; the RBF frontier curves to
enclose the classes. RBF is more flexible (and, with a large `gamma`, can overfit
— another bias/variance knob, like KNN's k).

## Head-to-head: every classifier in this part

Finally, the payoff — all four models (plus the logistic regression from the
[Regression](../02-regression/logistic-regression.ipynb) chapter) on the *same*
full 30-feature split, scored by accuracy and F1. An honest comparison: no model
is guaranteed to win.

In [ ]:
use smartcore::linear::logistic_regression::LogisticRegression;
use smartcore::neighbors::knn_classifier::{KNNClassifier, KNNClassifierParameters};
use smartcore::naive_bayes::gaussian::{GaussianNB, GaussianNBParameters};
use smartcore::api::SupervisedEstimator;

let rows: Vec<(String, f64, f64)> = {
    let (xtr, xte, ytr, yte) = train_test_split(&x_full, &y, 0.3, true, Some(42));
    // Fit each model in its own block, pushing out only the owned prediction Vec.
    let mut preds: Vec<(String, Vec<i32>)> = vec![];
    { let m = LogisticRegression::fit(&xtr, &ytr, Default::default()).unwrap();
      preds.push(("Logistic".into(), m.predict(&xte).unwrap())); }
    { let m = KNNClassifier::fit(&xtr, &ytr, KNNClassifierParameters::default().with_k(5)).unwrap();
      preds.push(("KNN (k=5)".into(), m.predict(&xte).unwrap())); }
    { // GaussianNB needs UNSIGNED labels; convert the split, predict, map back to i32.
      let ytr_u: Vec<u32> = ytr.iter().map(|&v| v as u32).collect();
      let m = GaussianNB::fit(&xtr, &ytr_u, GaussianNBParameters { priors: None }).unwrap();
      let pu: Vec<u32> = m.predict(&xte).unwrap();
      preds.push(("GaussianNB".into(), pu.iter().map(|&v| v as i32).collect())); }
    // SVC needs {-1,+1} labels and returns {-1,+1} as f32; map back to {0,1}.
    let ytr_svc: Vec<i32> = ytr.iter().map(|&v| if v == 1 { 1 } else { -1 }).collect();
    { let params = SVCParameters::default().with_c(1.0f32).with_kernel(Kernels::linear());
      let m = SVC::fit(&xtr, &ytr_svc, &params).unwrap();
      let pf: Vec<f32> = m.predict(&xte).unwrap();
      preds.push(("SVC (linear)".into(), pf.iter().map(|&v| if v > 0.0 { 1 } else { 0 }).collect())); }
    { let params = SVCParameters::default().with_c(1.0f32).with_kernel(Kernels::rbf().with_gamma(0.05));
      let m = SVC::fit(&xtr, &ytr_svc, &params).unwrap();
      let pf: Vec<f32> = m.predict(&xte).unwrap();
      preds.push(("SVC (RBF)".into(), pf.iter().map(|&v| if v > 0.0 { 1 } else { 0 }).collect())); }

    let yte_f: Vec<f32> = yte.iter().map(|&v| v as f32).collect();
    preds.iter().map(|(name, pr)| {
        let pf: Vec<f32> = pr.iter().map(|&v| v as f32).collect();
        (name.clone(), accuracy(&yte, pr), f1(&yte_f, &pf, 1.0))
    }).collect()
};

println!("{:<14} {:>9} {:>7}", "model", "accuracy", "f1");
println!("{}", "-".repeat(32));
for (name, acc, f1v) in &rows {
    println!("{:<14} {:>9.3} {:>7.3}", name, acc, f1v);
}

## Reading the comparison

On this dataset the numbers are close — which is itself the lesson: a good linear
model (logistic or linear SVM) is hard to beat on well-behaved tabular data, and
the "fancier" RBF SVM isn't automatically better without tuning `gamma`/`C`.
GaussianNB trails slightly, paying for its independence assumption, but stays in
the race for near-zero cost.

The right way to pick between close models — and to tune `C`, `gamma`, `k` — is
[cross-validation](../01d-evaluation/cross-validation.ipynb) plus a
[hyperparameter search](../05b-optimization/hyperparameter-search.ipynb), not a
single split. And a natural next question — *can combining these models beat any
one of them?* — is exactly what the [Ensemble](../04c-ensemble/random-forests.ipynb)
chapters take up.